In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
import community as community_louvain
from sklearn.metrics import silhouette_score, calinski_harabasz_score, pairwise_distances
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment

In [2]:
df = pd.read_csv("datasets/df_sberne.csv")

### Сначала построим кластеризацию, усредненную по годам

In [3]:
feature_cols = [
    'Здоровье_share',
    'Маркетплейсы_share',
    'Общественное питание_share',
    'Продовольствие_share',
    'Транспорт_share',
    'share_industry',
    'share_agriculture',
    'share_services',
    'urban_share',
    'Средняя зарплата',
    'population'
]

df_year = df.groupby(['region_name', 'mo', 'year'], as_index=False).agg({
    **{c: 'mean' for c in feature_cols},
})

print(df_year.shape)

(2055, 14)


In [4]:
df_2024 = df_year[df_year['year'] == 2024].copy()
df_2024 = df_2024.dropna(subset=feature_cols)
print(f"МО в 2024: {len(df_2024)}")

МО в 2024: 1026


In [5]:
X = df_2024[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
sim = cosine_similarity(X_scaled)

In [7]:
threshold = np.percentile(sim[sim > 0], 75)
print(f"Порог: {threshold:.3f}")
G = nx.Graph()

nodes = []

for _, row in df_2024.iterrows():
    node_id = (row['mo'], row['region_name'])
    nodes.append(node_id)

    G.add_node(
        node_id,
        mo=row['mo'],
        region_name=row['region_name'],
        population=row['population'],
        salary=row['Средняя зарплата'],
        urban=row['urban_share'],
        industry=row['share_industry'],
        agriculture=row['share_agriculture'],
        services=row['share_services'],
    )

for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        if sim[i, j] >= threshold:
            G.add_edge(
                nodes[i],
                nodes[j],
                weight=sim[i, j]
            )

print(f"Узлов: {G.number_of_nodes()}, рёбер: {G.number_of_edges()}")

Порог: 0.530
Узлов: 1026, рёбер: 63970


In [8]:
partition = community_louvain.best_partition(
    G,
    weight='weight',
    random_state=13,
    resolution = 0.9,
)

df_2024['node'] = list(zip(df_2024['mo'], df_2024['region_name']))
df_2024['cluster'] = df_2024['node'].map(partition)

print(df_2024['cluster'].value_counts())

cluster
4    298
1    226
0    215
2    176
3    111
Name: count, dtype: int64


### Кластеризация за 2024 год:

In [9]:
profile = df_2024.groupby('cluster')[feature_cols].mean().round(3)
profile['size'] = df_2024.groupby('cluster').size()
profile['share_of_total'] = (
    profile['size'] / len(df_2024) * 100
).round(1)
profile_df_2024 = profile.T
profile_df_2024.round(3)

cluster,0,1,2,3,4
Здоровье_share,0.050,0.049,0.047,0.039,0.046
Маркетплейсы_share,0.153,0.176,0.166,0.124,0.177
Общественное питание_share,0.049,0.018,0.030,0.033,0.016
Продовольствие_share,0.382,0.467,0.437,0.440,0.477
Транспорт_share,0.069,0.052,0.048,0.048,0.048
share_industry,0.234,0.105,0.397,0.168,0.079
share_agriculture,0.023,0.197,0.022,0.026,0.024
share_services,0.707,0.607,0.501,0.718,0.761
urban_share,0.672,0.294,0.744,0.709,0.495
Средняя зарплата,72995.526,53076.422,70984.624,107748.896,51629.587


In [10]:
df_2023 = df_year[df_year['year'] == 2023].copy()
df_2023 = df_2023.dropna(subset=feature_cols)
print(f"МО в {2023}: {len(df_2023)}")

МО в 2023: 1029


In [11]:
X2 = df_2023[feature_cols].values
scaler = StandardScaler()
X2_scaled = scaler.fit_transform(X2)

In [12]:
sim = cosine_similarity(X2_scaled)

In [13]:
threshold = np.percentile(sim[sim > 0], 75)
print(f"Порог: {threshold:.3f}")
G = nx.Graph()

nodes = []

for _, row in df_2023.iterrows():
    node_id = (row['mo'], row['region_name'])
    nodes.append(node_id)

    G.add_node(
        node_id,
        mo=row['mo'],
        region_name=row['region_name'],
        population=row['population'],
        salary=row['Средняя зарплата'],
        urban=row['urban_share'],
        industry=row['share_industry'],
        agriculture=row['share_agriculture'],
        services=row['share_services'],
    )

for i in range(len(nodes)):
    for j in range(i + 1, len(nodes)):
        if sim[i, j] >= threshold:
            G.add_edge(
                nodes[i],
                nodes[j],
                weight=sim[i, j]
            )

print(f"Узлов: {G.number_of_nodes()}, рёбер: {G.number_of_edges()}")

Порог: 0.524
Узлов: 1029, рёбер: 64339


In [14]:
partition = community_louvain.best_partition(
    G,
    weight='weight',
    random_state=13,
    resolution = 0.9,
)

df_2023['node'] = list(zip(df_2023['mo'], df_2023['region_name']))
df_2023['cluster'] = df_2023['node'].map(partition)

print(df_2023['cluster'].value_counts())

cluster
1    314
2    222
3    210
0    147
4    136
Name: count, dtype: int64


### Кластеризация за 2023 год:

In [15]:
profile = df_2023.groupby('cluster')[feature_cols].mean().round(3)
profile['size'] = df_2023.groupby('cluster').size()
profile['share_of_total'] = (
    profile['size'] / len(df_2023) * 100
).round(1)
profile_df_2023 = profile.T
profile_df_2023.round(3)

cluster,0,1,2,3,4
Здоровье_share,0.053,0.048,0.052,0.049,0.042
Маркетплейсы_share,0.111,0.130,0.130,0.121,0.089
Общественное питание_share,0.049,0.017,0.018,0.033,0.033
Продовольствие_share,0.392,0.497,0.486,0.449,0.451
Транспорт_share,0.073,0.053,0.057,0.054,0.051
share_industry,0.211,0.082,0.105,0.411,0.171
share_agriculture,0.028,0.025,0.200,0.021,0.023
share_services,0.730,0.748,0.603,0.524,0.724
urban_share,0.636,0.483,0.302,0.779,0.735
Средняя зарплата,62651.050,44024.508,45053.756,59715.565,86407.082


#### Теперь сопоставим кластеры 2023 года и 2024 года, используя Hungarian Algorithm.

In [16]:
X_2023 = profile_df_2023.loc[feature_cols].T
X_2024 = profile_df_2024.loc[feature_cols].T
scaler = StandardScaler()
X_all = pd.concat(
    [X_2023, X_2024],
    axis=0
)
scaler.fit(X_all)
X_2023_scaled = scaler.transform(X_2023)
X_2024_scaled = scaler.transform(X_2024)
distance = pairwise_distances(
    X_2023_scaled,
    X_2024_scaled,
    metric='euclidean'
)
row_ind, col_ind = linear_sum_assignment(distance)

cluster_mapping = pd.DataFrame({
    'cluster_2023': X_2023.index[row_ind],
    'cluster_2024': X_2024.index[col_ind],
    'distance': distance[row_ind, col_ind]
})

print(cluster_mapping)

   cluster_2023  cluster_2024  distance
0             0             0  2.002292
1             1             4  1.992385
2             2             1  2.027212
3             3             2  2.015539
4             4             3  1.912972


Получили хороший метчинг, растояние в среднем 2

In [17]:
mapping = dict(zip(cluster_mapping['cluster_2024'], cluster_mapping['cluster_2023']))

profile_df_2024_fixed = profile_df_2024.rename(
    columns=mapping
)

profile_df_2024_fixed = profile_df_2024_fixed.reindex(
    sorted(profile_df_2024_fixed.columns),
    axis=1
)

profile_df_2024_fixed.round(3)

cluster,0,1,2,3,4
Здоровье_share,0.050,0.046,0.049,0.047,0.039
Маркетплейсы_share,0.153,0.177,0.176,0.166,0.124
Общественное питание_share,0.049,0.016,0.018,0.030,0.033
Продовольствие_share,0.382,0.477,0.467,0.437,0.440
Транспорт_share,0.069,0.048,0.052,0.048,0.048
share_industry,0.234,0.079,0.105,0.397,0.168
share_agriculture,0.023,0.024,0.197,0.022,0.026
share_services,0.707,0.761,0.607,0.501,0.718
urban_share,0.672,0.495,0.294,0.744,0.709
Средняя зарплата,72995.526,51629.587,53076.422,70984.624,107748.896


In [18]:
profile_df_2023.round(3)

cluster,0,1,2,3,4
Здоровье_share,0.053,0.048,0.052,0.049,0.042
Маркетплейсы_share,0.111,0.130,0.130,0.121,0.089
Общественное питание_share,0.049,0.017,0.018,0.033,0.033
Продовольствие_share,0.392,0.497,0.486,0.449,0.451
Транспорт_share,0.073,0.053,0.057,0.054,0.051
share_industry,0.211,0.082,0.105,0.411,0.171
share_agriculture,0.028,0.025,0.200,0.021,0.023
share_services,0.730,0.748,0.603,0.524,0.724
urban_share,0.636,0.483,0.302,0.779,0.735
Средняя зарплата,62651.050,44024.508,45053.756,59715.565,86407.082


### Теперь делаем по месяцам

In [19]:
months = sorted(df[['year', 'month']].drop_duplicates().itertuples(index=False, name=None))
print(f"Всего месяцев: {len(months)}")

Всего месяцев: 24


In [20]:
RESOLUTION = 0.9
RANDOM_STATE = 13
THRESHOLD_PERCENTILE = 75

def cluster_one_month(df_m):
    df_m = df_m.dropna(subset=feature_cols).copy()
    X = df_m[feature_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    sim = cosine_similarity(X_scaled)
    threshold = np.percentile(sim[sim > 0], THRESHOLD_PERCENTILE)

    G = nx.Graph()
    nodes = []
    for _, row in df_m.iterrows():
        node_id = (row['mo'], row['region_name'])
        nodes.append(node_id)
        G.add_node(node_id)

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if sim[i, j] >= threshold:
                G.add_edge(nodes[i], nodes[j], weight=sim[i, j])

    partition = community_louvain.best_partition(
        G, weight='weight', random_state=RANDOM_STATE, resolution=RESOLUTION
    )

    df_m['node'] = list(zip(df_m['mo'], df_m['region_name']))
    df_m['cluster_raw'] = df_m['node'].map(partition)
    return df_m


monthly_data = {}
for (y, m) in months:
    df_m = df[(df['year'] == y) & (df['month'] == m)]
    df_clustered = cluster_one_month(df_m)
    monthly_data[(y, m)] = df_clustered
    print(f"{y}-{m:02d}: {len(df_clustered)} МО, {df_clustered['cluster_raw'].nunique()} кластеров")

2023-01: 1027 МО, 5 кластеров
2023-02: 1027 МО, 5 кластеров
2023-03: 1027 МО, 5 кластеров
2023-04: 1021 МО, 4 кластеров
2023-05: 1019 МО, 5 кластеров
2023-06: 1020 МО, 5 кластеров
2023-07: 1020 МО, 6 кластеров
2023-08: 1018 МО, 6 кластеров
2023-09: 1015 МО, 6 кластеров
2023-10: 1015 МО, 6 кластеров
2023-11: 1012 МО, 5 кластеров
2023-12: 1014 МО, 5 кластеров
2024-01: 1010 МО, 7 кластеров
2024-02: 1012 МО, 5 кластеров
2024-03: 1011 МО, 5 кластеров
2024-04: 961 МО, 5 кластеров
2024-05: 961 МО, 5 кластеров
2024-06: 958 МО, 4 кластеров
2024-07: 961 МО, 4 кластеров
2024-08: 963 МО, 6 кластеров
2024-09: 966 МО, 4 кластеров
2024-10: 956 МО, 5 кластеров
2024-11: 958 МО, 5 кластеров
2024-12: 963 МО, 4 кластеров


In [21]:
def get_profiles(df_clustered):
    return df_clustered.groupby('cluster_raw')[feature_cols].mean()


profiles = {}
for key, df_m in monthly_data.items():
    profiles[key] = get_profiles(df_m)

In [22]:
def match_profiles(profile_prev, profile_curr):
    common_cols = [c for c in profile_prev.columns if c in profile_curr.columns]
    X_prev = profile_prev[common_cols].values
    X_curr = profile_curr[common_cols].values

    scaler = StandardScaler()
    scaler.fit(np.vstack([X_prev, X_curr]))
    X_prev_s = scaler.transform(X_prev)
    X_curr_s = scaler.transform(X_curr)

    distance = pairwise_distances(X_prev_s, X_curr_s, metric='euclidean')
    row_ind, col_ind = linear_sum_assignment(distance)

    mapping = {}
    matched_curr = set()
    for r, c in zip(row_ind, col_ind):
        mapping[profile_curr.index[c]] = profile_prev.index[r]
        matched_curr.add(profile_curr.index[c])

    for c in profile_curr.index:
        if c not in matched_curr:
            nearest_idx = np.argmin(distance[:, c])
            mapping[c] = profile_prev.index[nearest_idx]

    return mapping


sorted_months = sorted(monthly_data.keys())
reference_month = sorted_months[0]

canonical_mapping = {}
canonical_mapping[reference_month] = {
    c: c for c in monthly_data[reference_month]['cluster_raw'].unique()
}

prof_reference = profiles[reference_month].rename(
    index=canonical_mapping[reference_month]
)

for curr_key in sorted_months[1:]:
    prof_curr = profiles[curr_key]
    mapping_curr = match_profiles(prof_reference, prof_curr)
    canonical_mapping[curr_key] = mapping_curr
    print(f"{reference_month} -> {curr_key}: кластеров: {len(mapping_curr)}")

profiles_canon = {}
for key in sorted_months:
    cmap = canonical_mapping[key]
    prof = profiles[key].copy()
    prof.index = prof.index.map(cmap)
    prof = prof.groupby(prof.index).mean()
    profiles_canon[key] = prof

match_distances = []

for prev_key, curr_key in zip(sorted_months[:-1], sorted_months[1:]):
    prof_a = profiles_canon[prev_key]
    prof_b = profiles_canon[curr_key]

    common = [c for c in prof_a.index if c in prof_b.index]
    a = prof_a.loc[common].values
    b = prof_b.loc[common].values

    scaler = StandardScaler()
    scaler.fit(np.vstack([a, b]))
    a_s = scaler.transform(a)
    b_s = scaler.transform(b)

    d = pairwise_distances(a_s, b_s, metric='euclidean')
    mean_dist = float(np.mean(np.diag(d)))

    match_distances.append({
        'month_prev': prev_key,
        'month_curr': curr_key,
        'mean_distance': mean_dist,
    })
    print(f"{prev_key} -> {curr_key}: mean distance = {mean_dist:.3f}")

(2023, 1) -> (2023, 2): кластеров: 5
(2023, 1) -> (2023, 3): кластеров: 5
(2023, 1) -> (2023, 4): кластеров: 4
(2023, 1) -> (2023, 5): кластеров: 5
(2023, 1) -> (2023, 6): кластеров: 5
(2023, 1) -> (2023, 7): кластеров: 6
(2023, 1) -> (2023, 8): кластеров: 6
(2023, 1) -> (2023, 9): кластеров: 6
(2023, 1) -> (2023, 10): кластеров: 6
(2023, 1) -> (2023, 11): кластеров: 5
(2023, 1) -> (2023, 12): кластеров: 5
(2023, 1) -> (2024, 1): кластеров: 7
(2023, 1) -> (2024, 2): кластеров: 5
(2023, 1) -> (2024, 3): кластеров: 5
(2023, 1) -> (2024, 4): кластеров: 5
(2023, 1) -> (2024, 5): кластеров: 5
(2023, 1) -> (2024, 6): кластеров: 4
(2023, 1) -> (2024, 7): кластеров: 4
(2023, 1) -> (2024, 8): кластеров: 6
(2023, 1) -> (2024, 9): кластеров: 4
(2023, 1) -> (2024, 10): кластеров: 5
(2023, 1) -> (2024, 11): кластеров: 5
(2023, 1) -> (2024, 12): кластеров: 4
(2023, 1) -> (2023, 2): mean distance = 0.976
(2023, 2) -> (2023, 3): mean distance = 0.692
(2023, 3) -> (2023, 4): mean distance = 2.506
(2023

In [23]:
prof_reference_canon = profiles_canon[sorted_months[0]]

prof_2023_for_match = profile_df_2023.loc[feature_cols].T
mapping_2023 = match_profiles(prof_reference_canon, prof_2023_for_match)
print("mapping_2023:", mapping_2023)

prof_2024_raw = profile_df_2024.loc[feature_cols].T
mapping_2024 = match_profiles(prof_reference_canon, prof_2024_raw)
print("mapping_2024:", mapping_2024)

df_2023['cluster_canon'] = df_2023['cluster'].map(mapping_2023)
df_2024['cluster_canon'] = df_2024['cluster'].map(mapping_2024)

df_2023['node'] = list(zip(df_2023['mo'], df_2023['region_name']))
df_2024['node'] = list(zip(df_2024['mo'], df_2024['region_name']))

mapping_2023: {np.int64(4): np.int64(0), np.int64(1): np.int64(1), np.int64(3): np.int64(2), np.int64(0): np.int64(3), np.int64(2): np.int64(4)}
mapping_2024: {np.int64(3): np.int64(0), np.int64(4): np.int64(1), np.int64(2): np.int64(2), np.int64(0): np.int64(3), np.int64(1): np.int64(4)}


In [24]:
for key, df_m in monthly_data.items():
    cmap = canonical_mapping[key]
    df_m['cluster_matched'] = df_m['cluster_raw'].map(cmap)

In [25]:
match_df = pd.DataFrame(match_distances)
print(match_df.to_string(index=False))
print(f"\nОбщая средняя distance: {match_df['mean_distance'].mean():.4f}")

month_prev month_curr  mean_distance
 (2023, 1)  (2023, 2)       0.975841
 (2023, 2)  (2023, 3)       0.691571
 (2023, 3)  (2023, 4)       2.506343
 (2023, 4)  (2023, 5)       1.956631
 (2023, 5)  (2023, 6)       1.859516
 (2023, 6)  (2023, 7)       0.821043
 (2023, 7)  (2023, 8)       1.199523
 (2023, 8)  (2023, 9)       2.222943
 (2023, 9) (2023, 10)       2.399091
(2023, 10) (2023, 11)       2.377343
(2023, 11) (2023, 12)       2.496521
(2023, 12)  (2024, 1)       2.895622
 (2024, 1)  (2024, 2)       1.701178
 (2024, 2)  (2024, 3)       0.807497
 (2024, 3)  (2024, 4)       0.842803
 (2024, 4)  (2024, 5)       1.176858
 (2024, 5)  (2024, 6)       2.524975
 (2024, 6)  (2024, 7)       1.097505
 (2024, 7)  (2024, 8)       2.617362
 (2024, 8)  (2024, 9)       2.352221
 (2024, 9) (2024, 10)       1.928828
(2024, 10) (2024, 11)       1.847599
(2024, 11) (2024, 12)       1.807223

Общая средняя distance: 1.7872


Дистанция хорошая, кластеры устойчивы во времени

In [29]:
cluster_distances = {c: [] for c in range(5)}

for prev_key, curr_key in zip(sorted_months[:-1], sorted_months[1:]):
    prof_a = profiles_canon[prev_key]
    prof_b = profiles_canon[curr_key]

    common = [c for c in prof_a.index if c in prof_b.index]
    if not common:
        continue

    a = prof_a.loc[common].values
    b = prof_b.loc[common].values

    scaler = StandardScaler()
    scaler.fit(np.vstack([a, b]))
    a_s = scaler.transform(a)
    b_s = scaler.transform(b)

    per_cluster = np.linalg.norm(a_s - b_s, axis=1)

    for c, dist in zip(common, per_cluster):
        cluster_distances[c].append(float(dist))

result_table = pd.DataFrame({
    'cluster': list(cluster_distances.keys()),
    'mean_distance': [np.mean(v) if v else np.nan for v in cluster_distances.values()]
}).sort_values('cluster').reset_index(drop=True)

result_table

,cluster,mean_distance
0,0,1.804655
1,1,1.407312
2,2,2.219526
3,3,2.482011
4,4,1.155853


In [30]:
all_months_df = pd.concat(
    [df_m.assign(month_key=f"{df_m['year'].iloc[0]}-{df_m['month'].iloc[0]:02d}")
     for df_m in monthly_data.values()],
    ignore_index=True
)
all_months_df['month_key'] = (
    all_months_df['year'].astype(str).str[-2:] + '-' +
    all_months_df['month'].astype(str).str.zfill(2)
)

wide_clusters = all_months_df.pivot_table(
    index=['region_name', 'mo'],
    columns='month_key',
    values='cluster_matched',
    aggfunc='first'
).reset_index()

wide_clusters.columns.name = None

year_2023 = (
    df_2023[['region_name', 'mo', 'cluster_canon']]
    .rename(columns={'cluster_canon': 'cluster_2023'})
)
year_2024 = (
    df_2024[['region_name', 'mo', 'cluster_canon']]
    .rename(columns={'cluster_canon': 'cluster_2024'})
)

wide_clusters = wide_clusters.merge(
    year_2023, on=['region_name', 'mo'], how='left'
).merge(
    year_2024, on=['region_name', 'mo'], how='left'
)


map_num = {
    0: 4,
    4: 2,
    1: 1,
    2: 3,
    3: 0
}

cluster_cols = [c for c in wide_clusters.columns if c not in ('region_name', 'mo')]
wide_clusters[cluster_cols] = wide_clusters[cluster_cols].apply(lambda s: s.map(map_num))
wide_clusters

,region_name,mo,23-01,23-02,23-03,23-04,23-05,23-06,23-07,23-08,...,24-05,24-06,24-07,24-08,24-09,24-10,24-11,24-12,cluster_2023,cluster_2024
0,Алтайский край,Алейский муниципальный район,NaN,NaN,NaN,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
1,Алтайский край,Алтайский муниципальный район,1.0,1.0,1.0,4.0,1.0,0.0,0.0,0.0,...,1.0,4.0,4.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
2,Алтайский край,Баевский муниципальный район,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,Алтайский край,Бийский муниципальный район,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
4,Алтайский край,Благовещенский муниципальный район,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1089,Ярославская область,Угличский муниципальный район,4.0,4.0,4.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
1090,Ярославская область,Ярославский муниципальный район,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,4.0,4.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0
1091,Ярославская область,городской округ город Переславль-Залесский,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
1092,Ярославская область,городской округ город Рыбинск,3.0,3.0,3.0,3.0,0.0,3.0,3.0,3.0,...,0.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0


In [28]:
wide_clusters.to_csv("final_clustering.csv", index=False)